# 🚀 Investment Assistant - Colab Pro SLM 服務端點

本筆記本專為 **Google Colab Pro (T4 / L4 / A100 GPU)** 與 **VS Code Colab Extension** 設計。
透過此 Notebook，您可以利用 Colab Pro 強大的 GPU 免費算力，將 `Qwen2.5-1.5B-Instruct`（或 7B）啟動為相容於 OpenAI 協定的專屬 API 伺服器，並自動透過 Cloudflare Tunnel 產生公開 HTTPS 端點，供本機的 Investment Assistant 進行即時自然語言意圖識別 (NLU) 與新聞快篩！

### 📌 執行步驟：
1. 在 VS Code 右上方核心選擇連線至 **Google Colab (GPU 執行階段，如 T4 或 L4)**。
2. 依序執行下方單元格。
3. 將產生的 `SLM_ENDPOINT_URL` 複製回本機的 `.env` 檔案。

### 1. 檢查 GPU 狀態

In [ ]:
!nvidia-smi

### 2. 安裝高效推論與穿透套件

In [ ]:
!pip install -q transformers accelerate bitsandbytes fastapi uvicorn pydantic pycloudflared requests httpx

### 3. 啟動 SLM 伺服器並自動開啟 Cloudflare 穿透端點

In [ ]:
import os
import time
import threading
import torch
from typing import List, Optional
from pydantic import BaseModel
import uvicorn
from fastapi import FastAPI, HTTPException
from transformers import pipeline, AutoTokenizer
from pycloudflared import try_cloudflare

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # 若為 A100/L4 亦可選 "Qwen/Qwen2.5-7B-Instruct"
PORT = 8000

app = FastAPI(title="Colab Pro SLM Serving Endpoint")

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: str
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.1
    max_tokens: Optional[int] = 256

@app.get("/health")
def health():
    return {"status": "ok", "device": "cuda" if torch.cuda.is_available() else "cpu", "model": MODEL_ID}

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model", "owned_by": "colab-pro"}]}

@app.post("/v1/chat/completions")
async def chat_completions(req: ChatCompletionRequest):
    conversation = [{"role": m.role, "content": m.content} for m in req.messages]
    try:
        outputs = model_pipeline(
            conversation,
            max_new_tokens=req.max_tokens or 256,
            temperature=req.temperature or 0.1,
            do_sample=req.temperature > 0.0,
            pad_token_id=tokenizer.eos_token_id,
        )
        response_text = outputs[0]["generated_text"][-1]["content"]
        return {
            "id": f"chatcmpl-{int(time.time())}",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": MODEL_ID,
            "choices": [{"index": 0, "message": {"role": "assistant", "content": response_text.strip()}, "finish_reason": "stop"}],
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print(f"[Colab Pro] Loading model {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_pipeline = pipeline(
    "text-generation",
    model=MODEL_ID,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.float16,
)
print("[Colab Pro] Model loaded successfully!")

# 啟動 Cloudflare 穿透
tunnel = try_cloudflare(port=PORT)
public_url = tunnel.tunnel.replace("http://", "https://")
print("\n" + "="*65)
print("🎉 [Colab Pro] 專屬 SLM API 端點已成功上線！")
print(f"🔗 公開端點 URL: {public_url}")
print("="*65)
print("請將以下參數複製填入您本地專案的 .env 檔案：")
print(f"SLM_ENABLED=true")
print(f"SLM_ENDPOINT_URL={public_url}/v1")
print(f"SLM_MODEL={MODEL_ID}")
print("="*65 + "\n")

# 啟動 Web 服務
uvicorn.run(app, host="0.0.0.0", port=PORT)

### 4. 測試端點推論品質 (可在 Colab 新開單元格或終端機測試)

In [ ]:
import requests

test_payload = {
    "model": MODEL_ID,
    "messages": [
        {"role": "system", "content": "你是一位台股投資助理 NLU 解析器，請將句子轉為純 JSON。"},
        {"role": "user", "content": "剛剛在 1050 買進 8 股台積電"}
    ]
}

res = requests.post(f"http://localhost:{PORT}/v1/chat/completions", json=test_payload)
print("Status:", res.status_code)
print("Response:", res.json()["choices"][0]["message"]["content"])